# 第03回 機械学習1：値とコードを一段ずつ読む

同じ固定データを使い、作る・選ぶ・評価するの三つを分けます。各事例の予測と正解数を自分で数えます。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** しきい値モデルは学習7/8・検証3/4、記憶モデルは学習8/8・検証2/4。検証でしきい値モデルを選ぶ。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
from fractions import Fraction


In [ ]:
import json


In [ ]:
TRAIN = (
    ("tr01", 10, 0), ("tr02", 20, 0), ("tr03", 30, 1), ("tr04", 40, 0),
    ("tr05", 60, 1), ("tr06", 70, 1), ("tr07", 80, 1), ("tr08", 90, 1),
)


In [ ]:
VALIDATION = (
    ("va01", 15, 0), ("va02", 35, 0), ("va03", 65, 1), ("va04", 85, 1),
)


In [ ]:
TEST = (
    ("te01", 5, 0), ("te02", 25, 0), ("te03", 75, 1), ("te04", 95, 1),
)


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 事例の役割

**今いる段階：** 分割→用途

**予想：** どの事例を作成に使うか

**操作：** example_partを'training'から'test'へ変更し、同じセルを実行します。

**結果の目印：** 学習8・検証4・テスト4

**戻す：** example_part='training'に戻して再実行します。

**説明する：** 評価を選択に使わない理由


### 計算の定義：`lesson_splits`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_splits(part='training'):
    splits = {'training':TRAIN,'validation':VALIDATION,'test':TEST}
    roles = {'training':'モデルを作る','validation':'モデルを選ぶ','test':'選択後に評価する'}
    rows = [(part,*row,roles[part]) for row in splits[part]]
    return {'title':'どの事例を何に使うか','question':'テストを見てモデルを選び直してよいか',
            'input':part,'columns':['区分','ID','特徴','正解','使い道'],'rows':rows,
            'tables':[('全体の分割',['区分','件数','ID','役割'],[(key,len(value),[r[0] for r in value],roles[key]) for key,value in splits.items()])],
            'result':'学習8件→検証4件で選択→テスト4件で評価。IDに重複なし。',
            'comparison':'partをtraining→testで、モデルを作る事例と未知評価の事例を比べる。',
            'explanation':'ラベルは学習や答え合わせに使う。predictへ渡すのは特徴だけ。IDが異なるだけで漏洩が完全に防げるわけではない。',
            'reset':'part="training"で再実行。'}


In [ ]:
example_part = 'training'
program_observation_splits = lesson_splits(example_part)
learning_show(program_observation_splits, reset="example_part='training'に戻して再実行します。")


## 2. 学習中の候補比較

**今いる段階：** 候補→予測→誤り→採用

**予想：** どこに境界を置くか

**操作：** example_labelを1から0へ変更し、同じセルを実行します。

**結果の目印：** 基準はしきい値25・学習誤り1

**戻す：** example_label=1に戻して再実行します。

**説明する：** 同点時の規約


### 計算の定義：`validate_rows`

この教材の入力条件を確認する。ラベルの正しさ自体は保証しない。


In [ ]:
def validate_rows(rows):
    """この教材の入力条件を確認する。ラベルの正しさ自体は保証しない。"""
    if not rows:
        raise ValueError("データは1件以上必要です")
    ids = set()
    for row in rows:
        if len(row) != 3:
            raise ValueError("各行は (ID, 特徴, ラベル) にしてください")
        identifier, feature, label = row
        if not isinstance(identifier, str) or not identifier or identifier in ids:
            raise ValueError("IDは空でない文字列で、重複しないようにしてください")
        if type(feature) is not int or type(label) is not int or label not in (0, 1):
            raise ValueError("特徴は整数、ラベルは整数の0または1にしてください")
        ids.add(identifier)


### 計算の定義：`fit_threshold`

学習誤り最小のしきい値を学習。同点は小さいしきい値を採用。


In [ ]:
def fit_threshold(train, record=None):
    """学習誤り最小のしきい値を学習。同点は小さいしきい値を採用。"""
    validate_rows(train)
    values = sorted({x for _, x, _ in train})
    candidates = [Fraction(values[0] - 1)]
    candidates += [Fraction(a + b, 2) for a, b in zip(values, values[1:])]
    candidates += [Fraction(values[-1] + 1)]
    scored = []
    for candidate in candidates:
        errors = 0
        for _, feature, label in train:
            predicted = int(feature >= candidate)
            errors += predicted != label
        scored.append((errors, candidate))
        if record is not None:
            record.append((candidate, errors, len(train)-errors))
    _, threshold = min(scored)
    return {"threshold": threshold}


### 計算の定義：`lesson_threshold`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_threshold(label_tr03=1):
    train = [tuple(row) for row in TRAIN]
    train[2] = (train[2][0],train[2][1],label_tr03)
    trace = []
    model = fit_threshold(train,record=trace)
    rows = [(str(t),errors,correct,'採用' if t==model['threshold'] else '') for t,errors,correct in trace]
    return {'title':'候補のしきい値を実際に比較する','question':'学習データを全部正解にできるか',
            'input':train,'columns':['しきい値候補','誤り数/8','正解数/8','選択'],'rows':rows,
            'result':str(model['threshold']),'comparison':'tr03の正解を1→0に変えると、学習誤りは1→0。原データは保持。',
            'explanation':'予測は特徴≥しきい値なら1。隣り合う特徴の中点と両端の候補を調べ、誤りが最少、同点なら小さいしきい値を選ぶ。',
            'reset':'label_tr03=1で再実行。TRAINのコピーから開始。'}


In [ ]:
example_label = 1
program_observation_threshold = lesson_threshold(example_label)
learning_show(program_observation_threshold, reset='example_label=1に戻して再実行します。')


## 3. 各事例の予測

**今いる段階：** 特徴→予測→正解と比較

**予想：** 学習で全問正解なら十分か

**操作：** example_modelを'threshold'から'memory'へ変更し、同じセルを実行します。

**結果の目印：** 検証では記憶モデル2/4

**戻す：** example_model='threshold'に戻して再実行します。

**説明する：** 予測の入力に正解を含めない理由


### 計算の定義：`fit_memory`

特徴とラベルを記憶。未知の特徴は学習多数派、同数なら0。


In [ ]:
def fit_memory(train):
    """特徴とラベルを記憶。未知の特徴は学習多数派、同数なら0。"""
    validate_rows(train)
    labels = {}
    for _, x, label in train:
        if x in labels and labels[x] != label:
            raise ValueError("記憶モデルでは同じ特徴に相反するラベルを記憶できません")
        labels[x] = label
    ones = sum(label for _, _, label in train)
    return {"labels": labels, "fallback": int(ones > len(train) - ones)}


### 計算の定義：`predict`

入力の特徴だけから予測。正解ラベルは受け取らない。


In [ ]:
def predict(model, feature):
    """入力の特徴だけから予測。正解ラベルは受け取らない。"""
    if type(feature) is not int:
        raise ValueError("予測する特徴は整数にしてください")
    if "threshold" in model:
        return int(feature >= model["threshold"])
    return model["labels"].get(feature, model["fallback"])


### 計算の定義：`evaluate`

予測してからラベルと比較。正解数と総数をそのまま返す。


In [ ]:
def evaluate(model, rows):
    """予測してからラベルと比較。正解数と総数をそのまま返す。"""
    validate_rows(rows)
    predictions = [predict(model, x) for _, x, _ in rows]
    correct = sum(prediction == row[2] for prediction, row in zip(predictions, rows))
    return {"predictions": predictions, "correct": correct, "total": len(rows)}


### 計算の定義：`lesson_predictions`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_predictions(model_name='threshold'):
    model = fit_threshold(TRAIN) if model_name=='threshold' else fit_memory(TRAIN)
    rows = []
    for part,data in [('学習',TRAIN),('検証',VALIDATION),('テスト',TEST)]:
        score = evaluate(model,data)
        for row,predicted in zip(data,score['predictions']):
            rows.append((part,row[0],row[1],predicted,row[2],predicted==row[2]))
    return {'title':'予測してから正解と照合する','question':'記憶した事例での全問正解は未知にも通じるか',
            'input':f'モデル={model_name}、学習データのみから作成。',
            'columns':['区分','ID','渡す特徴','予測','正解','一致'],'rows':rows,
            'result':{name:evaluate(model,data) for name,data in [('学習',TRAIN),('検証',VALIDATION),('テスト',TEST)]},
            'comparison':'threshold→memoryで学習は7/8→8/8、検証は3/4→2/4。',
            'explanation':'未知の特徴を記憶モデルは多数派1で予測する。学習の正解数だけで良さを判断しない。',
            'reset':'model_name="threshold"で再実行。'}


In [ ]:
example_model = 'threshold'
program_observation_predictions = lesson_predictions(example_model)
learning_show(program_observation_predictions, reset="example_model='threshold'に戻して再実行します。")


## 4. 選択と未知評価

**今いる段階：** 検証集計→選択→テスト

**予想：** 選択に使うのはどの区分か

**操作：** example_validation_labelを0から1へ変更し、同じセルを実行します。

**結果の目印：** 検証を変えて比較し、テストは固定

**戻す：** example_validation_label=0に戻して再実行します。

**説明する：** 検証とテストの違い


### 計算の定義：`validate_splits`

同じIDの使い回しを検出する。別IDの類似データ漏洩は別途点検が必要。


In [ ]:
def validate_splits(train, validation, test):
    """同じIDの使い回しを検出する。別IDの類似データ漏洩は別途点検が必要。"""
    seen = set()
    for rows in (train, validation, test):
        validate_rows(rows)
        ids = {row[0] for row in rows}
        if seen & ids:
            raise ValueError("学習・検証・テストで同じIDを使わないでください")
        seen.update(ids)


### 計算の定義：`choose_model`

検証結果で選択。同点ならthreshold。テスト結果は受け取らない。


In [ ]:
def choose_model(validation_scores):
    """検証結果で選択。同点ならthreshold。テスト結果は受け取らない。"""
    def accuracy(name):
        score = validation_scores[name]
        return Fraction(score["correct"], score["total"])
    return "threshold" if accuracy("threshold") >= accuracy("memory") else "memory"


### 計算の定義：`run_experiment`

学習→検証で選択→固定してテスト。原データは変更しない。


In [ ]:
def run_experiment(train=TRAIN, validation=VALIDATION, test=TEST):
    """学習→検証で選択→固定してテスト。原データは変更しない。"""
    validate_splits(train, validation, test)
    models = {"threshold": fit_threshold(train), "memory": fit_memory(train)}
    scores = {name: {
        "training": evaluate(model, train),
        "validation": evaluate(model, validation),
    } for name, model in models.items()}
    selected = choose_model({name: score["validation"] for name, score in scores.items()})
    # 選択を確定した後、比較教材として両モデルのテスト結果を表示する。
    # この表示を見てモデルや設定を選び直さない。
    for name, model in models.items():
        scores[name]["test"] = evaluate(model, test)
    scores["threshold"]["threshold"] = str(models["threshold"]["threshold"])
    scores["memory"]["fallback"] = models["memory"]["fallback"]
    return {
        "data": {"training": [list(row) for row in train],
                 "validation": [list(row) for row in validation],
                 "test": [list(row) for row in test]},
        "models": scores,
        "selected_model": selected,
        "selected_test": scores[selected]["test"],
    }


### 計算の定義：`lesson_selection`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_selection(validation_label=0):
    validation = list(VALIDATION)
    validation[0] = (validation[0][0],validation[0][1],validation_label)
    result = run_experiment(validation=validation)
    rows = []
    for name,scores in result['models'].items():
        for part in ('training','validation','test'):
            s=scores[part]
            rows.append((name,part,s['correct'],s['total'],s['correct']/s['total']))
    return {'title':'検証で選択を確定してからテスト','question':'どの正解数を選択に使ったか',
            'input':f'検証va01のラベル={validation_label}、テストは固定。',
            'columns':['モデル','区分','正解数','分母','正解率'],'rows':rows,
            'result':{'選んだモデル':result['selected_model'],'選択後のテスト':result['selected_test']},
            'comparison':'検証ラベルを変えてもテストデータを使ってモデルを選び直さない。',
            'explanation':'両モデルのテスト結果は比較教材として表示するが、選択処理choose_modelは検証スコアだけを受け取る。',
            'reset':'validation_label=0で再実行。検証データをコピーから戻す。'}


In [ ]:
example_validation_label = 0
program_observation_selection = lesson_selection(example_validation_label)
learning_show(program_observation_selection, reset='example_validation_label=0に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`print_result`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def print_result(result):
    names = {"threshold": "しきい値モデル", "memory": "記憶モデル"}
    print("架空の固定データ：学習8件・検証4件・テスト4件")
    print("学習したしきい値:", result["models"]["threshold"]["threshold"])
    for name, scores in result["models"].items():
        parts = [f"{title} {scores[key]['correct']}/{scores[key]['total']}"
                 for key, title in (("training", "学習"), ("validation", "検証"), ("test", "テスト"))]
        print(names[name] + ": " + " / ".join(parts))
    print("検証で選択:", names[result["selected_model"]])
    print("テスト結果を見て選び直さず、未知のデータでの限界を読みます。")


## 1. データと処理コード

架空の整数の特徴と0/1ラベルを使います。IDは同じ観測を別の用途に使い回さないための印です。学習8件・検証4件・テスト4件は固定され、乱数は使いません。

しきい値モデルは、学習誤りが最少の候補を採用し、同点なら小さい値を採用します。記憶モデルは既知の特徴のラベルを記憶し、未知の特徴には学習データの多数派ラベルを返します。同数なら0です。特徴とラベルの対応や比率は説明のために作ったもので、実データの性能を表しません。


## 2. 基準結果を確かめる

学習データだけでモデルを作り、検証の正解率で選びます。同点ならしきい値モデルを採用します。選択を確定した後でテストします。比較教材として両モデルのテスト結果も表示しますが、それを見て選び直してはいけません。

予想してから実行しましょう。学習時に全問正解したモデルが、検証でも優れるでしょうか？


In [ ]:
baseline_result = run_experiment()
print_result(baseline_result)


## 3. 予測列と誤りを読む

想定結果：しきい値は25。しきい値モデルは学習7/8、検証3/4、テスト3/4。記憶モデルは学習8/8、検証2/4、テスト2/4。検証でしきい値モデルを選び、その最終テストは3/4です。

しきい値モデルは検証の特徴35とテストの特徴25を誤ります。25以上を1にするためです。記憶モデルは未知の特徴をすべて多数派の1にし、検証・テストともラベル0の2件を誤ります。これは特定の架空例です。「この手法なら常に75%」とは言えません。


In [ ]:
for name, scores in baseline_result["models"].items():
    print(name)
    for split in ("training", "validation", "test"):
        score = scores[split]
        print(split, score["predictions"],
              "正解率", Fraction(score["correct"], score["total"]))


## 4. 入力だけを変える

学習済みモデルを固定したまま24、25、26を予測します。しきい値モデルは `[0, 1, 1]`、記憶モデルは未知の値なので `[1, 1, 1]` になります。これは推論です。入力を変えて予測するだけでは再学習しません。


In [ ]:
threshold_model = fit_threshold(TRAIN)
memory_model = fit_memory(TRAIN)
new_inputs = [24, 25, 26]  # このリストの値を変えて比較
print("しきい値:", [predict(threshold_model, x) for x in new_inputs])
print("記憶:", [predict(memory_model, x) for x in new_inputs])


## 5. 学習ラベルを一つ変える

元データは保持し、tr03のラベルを1から0へ変えた別の学習例を作ります。新たなしきい値は50、学習正解数は8/8、検証正解数は4/4になります。ラベルの変更が学習結果を変えることを観察します。実データでこの変更が正しいという意味ではありません。

変更例は学習・検証だけで観察します。すでに見た最終テストを繰り返し見ながら改善する手順にはしません。


In [ ]:
changed_train = [(identifier, x, 0 if identifier == "tr03" else label)
                 for identifier, x, label in TRAIN]
changed_model = fit_threshold(changed_train)
print("変更後のしきい値:", changed_model["threshold"])
print("学習:", evaluate(changed_model, changed_train))
print("検証:", evaluate(changed_model, VALIDATION))
print("元のtr03:", TRAIN[2])


## 6. 学習データの構成を変える

tr03を除いた学習集合をコピーして比較します。しきい値は50になります。記憶モデルはラベル0が3件、1が4件なので未知入力に1を返します。

行の追加・削除を試すときはIDを重複させず、検証・テストの観測を学習側へ移して評価を良く見せないでください。この検査は同一IDの重複を検出するだけで、別IDにした同じ対象や似た記録を自動的に見抜くものではありません。


In [ ]:
smaller_train = [row for row in TRAIN if row[0] != "tr03"]
validate_splits(smaller_train, VALIDATION, TEST)
for name, model in (("しきい値", fit_threshold(smaller_train)),
                    ("記憶", fit_memory(smaller_train))):
    print(name, model)
    print("検証:", evaluate(model, VALIDATION))


## まとめ

1. どの値がデータから決まり、どの値が人の決めた規則でしたか？
2. 記憶モデルの学習正解率100%だけで、未知の入力への対応を判断できますか？
3. 検証用と最終テスト用の役割はどう違いますか？

正解数と総数で比較し、有理数で正解率を計算しました。浮動小数点の誤差や乱数はありません。Python版とこのNotebookの基準データ・予測列・正解数・選択結果は完全一致で検証します。学生の実行環境や授業での理解度は別に確認が必要です。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python generalization.py --explain all`で同じ計算表を表示します。各テーマは`--explain splits`のように指定できます。従来のコマンドと固定期待値も維持しています。
